In [92]:
import os
import nibabel as nib
import numpy as np

In [93]:
# Path to your affine transformation file (sourced from defaced step)
transform_file = "./data/example_output/defaced_images/IXI002-Guys-0828-T1/IXI002-Guys-0828-T1.txt"

# The image you want to register (T1-w atlas image)
moving_image = "./data/icbm152_ext55_model_sym_2020_nifti/icbm152_ext55_model_sym_2020/mni_icbm152_t1_tal_nlin_sym_55_ext.nii"

# The reference space (defaced image)
reference_image = "./data/example_output/defaced_images/IXI002-Guys-0828-T1/IXI002-Guys-0828-T1_masked.nii.gz"

# Output path for the registered image
output_image = "atlas_registered_to_test.nii.gz"

# path to BRAINSFit executable
BRAINSFit_bin_path = os.path.join("./BRAINSTools/BRAINSFit")

# path to BRAINSFit executable
BRAINSresample_bin_path = "./BRAINSTools/BRAINSResample"

# Apply the transformation using BRAINSresample
os.system(f'"{BRAINSresample_bin_path}" ' +
          f'--inputVolume "{moving_image}" ' +
          f'--referenceVolume "{reference_image}" ' +
          f'--outputVolume "{output_image}" ' +
          f'--warpTransform "{transform_file}" ' +
          f'--inverseTransform ' +
          f'--interpolationMode Linear')

0

In [94]:
# Load the registered output and reference images
output_nii = nib.load(output_image)
output_data = output_nii.get_fdata()
reference_nii = nib.load(reference_image)
reference_data = reference_nii.get_fdata()

print(f"Data type check:")
print(f"  Reference data type: {reference_nii.get_data_dtype()}")
print(f"  Registered atlas data type: {output_nii.get_data_dtype()}")

# Simple linear intensity normalisation (mri_reface approach)
ref_brain_mask = reference_data > 0
output_brain_mask = output_data > 0

ref_brain_voxels = reference_data[ref_brain_mask]
output_brain_voxels = output_data[output_brain_mask]

ref_mean = np.mean(ref_brain_voxels)
ref_std = np.std(ref_brain_voxels)
output_mean = np.mean(output_brain_voxels)
output_std = np.std(output_brain_voxels)

print(f"BEFORE normalisation:")
print(f"  Reference (defaced): mean={ref_mean:.2f}, std={ref_std:.2f}, range=[{reference_data.min():.2f}, {reference_data.max():.2f}]")
print(f"  Registered Atlas: mean={output_mean:.2f}, std={output_std:.2f}, range=[{output_data.min():.2f}, {output_data.max():.2f}]")

# Apply linear transform (((v(x,y,z) - mean_atlas) / std_atlas) * std_ref ) + mean_ref
normalized_output_data = ((output_data - output_mean) / output_std) * ref_std + ref_mean

print(f"\nAFTER normalisation:")
print(f"  Reference (defaced): mean={ref_mean:.2f}, std={ref_std:.2f}, range=[{reference_data.min():.2f}, {reference_data.max():.2f}]")
print(f"  Registered Atlas (normalised): mean={np.mean(normalized_output_data[output_brain_mask]):.2f}, std={np.std(normalized_output_data[output_brain_mask]):.2f}, range=[{normalized_output_data.min():.2f}, {normalized_output_data.max():.2f}]")

# Handle data type conversion (AFTER normalisation)
reference_dtype = reference_nii.get_data_dtype()
print(f"\nConverting to reference data type: {reference_dtype}")

if reference_dtype in [np.uint8, np.uint16, np.uint32]:
    # For unsigned integers, clip to valid range and round
    normalized_output_data = np.clip(normalized_output_data, 0, np.iinfo(reference_dtype).max)
    normalized_output_data = np.round(normalized_output_data)
elif reference_dtype in [np.int8, np.int16, np.int32]:
    # For signed integers, clip to valid range and round
    normalized_output_data = np.clip(normalized_output_data, 
                            np.iinfo(reference_dtype).min, 
                            np.iinfo(reference_dtype).max)
    normalized_output_data = np.round(normalized_output_data)
# For float types, no rounding needed

print(f"Data type check:")
print(f"  Output data type: {output_nii.get_data_dtype()}")
print(f"  Reference data type: {reference_nii.get_data_dtype()}")


# Save the intensity-normalized output
normalized_output_nii = nib.Nifti1Image(normalized_output_data, affine=output_nii.affine)
nib.save(normalized_output_nii, output_image.replace(".nii.gz", "_normalised.nii.gz"))
print(f"\nIntensity normalization applied. Output saved to {output_image}")

Data type check:
  Reference data type: float64
  Registered atlas data type: float64
BEFORE normalisation:
  Reference (defaced): mean=200.46, std=181.06, range=[0.00, 1068.00]
  Registered Atlas: mean=34.10, std=34.11, range=[-0.01, 168.97]

AFTER normalisation:
  Reference (defaced): mean=200.46, std=181.06, range=[0.00, 1068.00]
  Registered Atlas (normalised): mean=200.46, std=181.06, range=[19.39, 916.44]

Converting to reference data type: float64
Data type check:
  Output data type: float64
  Reference data type: float64

Intensity normalization applied. Output saved to atlas_registered_to_test.nii.gz


In [95]:
# Create merged image
merged_data = reference_data.copy()

# Where reference is 0, fill with normalized atlas
mask_to_fill = reference_data == 0
merged_data[mask_to_fill] = normalized_output_data[mask_to_fill]

# Save the merged image
merged_nii = nib.Nifti1Image(merged_data, affine=reference_nii.affine)
final_image = output_image.replace(".nii.gz", "_refaced.nii.gz")
nib.save(merged_nii, final_image)

print(f"Merged image saved to {final_image}")
print(f"Voxels filled from atlas: {np.sum(mask_to_fill)}")
print(f"Total voxels: {mask_to_fill.size}")

Merged image saved to atlas_registered_to_test_refaced.nii.gz
Voxels filled from atlas: 5094456
Total voxels: 9830400


### Defacing with Afni - commands

In [96]:
# Get face region
# On the terminal (needs to be bash)
# @afni_refacer_run -input atlas_registered_to_test_refaced.nii.gz -mode_deface -prefix brats_defaced

In [97]:
# Get defaced scan
# on the terminal (needs to be bash)
# 3dcalc \
#     -a atlas_registered_to_test_refaced.nii.gz \
#     -b brats_defaced+orig \
#     -expr 'a*b' \
#     -prefix final_defaced.nii.gz